In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

以下代码要求有可以访问的聚源数据库，且设置好了配置文件。或者执行 [import_postgres_jydb_demo_data.py](../tools/import_postgres_jydb_demo_data.py) 脚本生成示例数据，这要求有写入权限的 PostgreSQL 数据库。

# JYDB

`JYDB`（聚源因子库）是基于关系数据库（PostgreSQL / MySQL / SQL Server / Oracle）构建的因子库，通过 SQL 查询来完成因子的数据访问。

## 存储映射

JYDB 将关系数据库映射为因子模型：

| 逻辑层 | 存储层 |
|--------|--------|
| 因子库（JYDB） | 整个数据库 |
| 因子表（FactorTable） | 数据库中的一张表 |
| 因子（Factor） | 数据库表中的一个字段 |

本质上，JYDB 将数据库表的多列数据重组为三维因子数据（因子 × 时点 × 证券代码）。因子表的类型（`TableType`）决定了这个重组的具体方式。

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库<br/>JYDB]
        B1[因子表<br/>FactorTable]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子1]
        F2[因子2]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[关系数据库]
        D1[数据库表]
        D2[数据库表]
        C --> D1
        C --> D2
        E2[字段2]
        E1[字段1]
        D2 --> E1
        D2 --> E2
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E1

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
```

## 因子表类型概览

JYDB 的因子表根据底层数据库表的数据组织形式，分为多种类型。每种类型通过 `TableType` 参数指定（通过 `getTable` 的 `args` 传入），决定了如何将关系数据库的行列数据映射为三维因子数据。

| 类型标识 | 类 | 数据库表特征 | 典型示例 |
|----------|-----|-------------|----------|
| `WideTable` | `_WideTable` | 一行 = 一个 ID-时点组合，每个字段是一个因子 | `日行情表` |
| `FeatureTable` | `_FeatureTable` | 一行 = 一个 ID，每个字段是一个因子（无时点维度或只取最新值） | `A股证券主表` |
| `TimeSeriesTable` | `_TimeSeriesTable` | 无 ID 字段，只有时点字段；每个时点一行，所有 ID 填充同样的值 | 宏观经济指标表 |
| `NarrowTable` | `_NarrowTable` | 高瘦表：一行 = 一个（ID, 时点, 因子名, 因子值）四元组 | — |
| `MappingTable` | `_MappingTable` | 起始时点-截止时点范围填充 | `公司行业划分表` |
| `ConstituentTable` | `_ConstituentTable` | 成份股 0/1 标记，按类别字段展开为多个因子 | `指数成份` |
| `FinancialTable` | `_FinancialTable` | 含报告期/公告日期的财务数据，支持最新/单季度/TTM 计算 | `资产负债表_新会计准则` |
| `FinancialIndicatorTable` | `_FinancialIndicatorTable` | 财务衍生指标，继承自 FinancialTable | — |
| `AnalystConsensusTable` | `_AnalystConsensusTable` | 分析师一致预期数据 | — |
| `AnalystEstDetailTable` | `_AnalystEstDetailTable` | 分析师明细预测数据 | — |
| `AnalystRatingDetailTable` | `_AnalystRatingDetailTable` | 分析师评级明细数据 | — |

如果不显式指定 `TableType`，JYDB 会根据库信息文件（`JYDBInfo.xlsx`）中预定义的映射自动选择。

In [3]:
# 创建因子库对象并 connect
from QuantStudio.Factor.JYDB import JYDB

FDB = JYDB().connect()
print(qs_help(FDB))

类型: JYDB
模块: QuantStudio.Factor.JYDB
QS 对象类型: 因子库
QS 对象名称: JYDB
QSID: ac23d89cc807bc8e0ba73303b717d01a1f8a10add7538df51af6edab0ed55099
参数集:
    * Name(名称): <class 'str'>, 默认值 'JYDB', 当前取值: 'JYDB'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'JYDB'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: '25.tcp.cpolar.top'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 10274
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'shzq'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前取值: 3
    * ConnIntervalSeconds(连接重试间隔): <clas

In [4]:
# 获取可用的因子表列表
print(FDB.TableNames[:5])

['交易日表(新)', '人员表', '行业表', '行业类别表', 'A股证券主表']


# 时点与 ID 获取

JYDB 提供了一系列方法直接查询数据库获取时点序列和证券代码序列，无需先获取因子表。

## 交易日序列

In [ ]:
# 获取指定交易所和日期范围的交易日
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 5))
print(DTs)

## 证券代码序列

JYDB 支持多种证券类型的 ID 查询：

| 方法 | 证券类型 | 后缀格式 |
|------|----------|----------|
| `getStockID` | 股票（A股/港股/美股/三板） | `.SZ` / `.SH` / `.BJ` / `.HK` / `.A` / `.O` / `.N` / `.NQ` |
| `getMutualFundID` | 公募基金 | `.OF` |
| `getBondID` | 债券 | 取决于交易所 |
| `getFutureID` | 期货 | `.CFE` / `.SHF` 等 |
| `getFutureCode` | 期货品种代码 | 品种代码如 `IF`、`A` 等 |
| `getOptionID` | 期权 | 如 `510050C1503M02200` |
| `getIndustryID` | 行业分类 | 行业代码 |

各方法共享的核心参数：
| 参数 | 说明 |
|------|------|
| `date` | 指定日期，`None` 表示当前日期 |
| `is_current` | `True`（默认）：截至指定日仍在市；`False`：历史上曾在市的所有证券 |
| `start_date` | 起始日。配合 `is_current`：`True` 表示区间内始终在市；`False` 表示区间内曾在市 |

In [ ]:
# 获取全体A股（含已退市）
IDs = FDB.getStockID(is_current=False)
print(IDs[:5])

In [ ]:
# 获取全体公募基金（含已清盘）
IDs = FDB.getMutualFundID(is_current=False)
print(IDs[:5])

In [ ]:
# 获取股指期货（中金所 IF 品种）
IDs = FDB.getFutureID(exchange="CFFEX", future_code="IF", is_current=False)
print(IDs[:5])

In [ ]:
# 获取期货品种代码
IDs = FDB.getFutureCode(exchange=("SHFE", "INE", "DCE", "CZCE", "GFEX", "CFFEX"), is_current=False)
print(IDs[:5])

In [ ]:
# 获取上证50ETF期权代码
IDs = FDB.getOptionID(option_code="510050", is_current=False)
print(IDs[:5])

## 读取数据

因子表的 `readData` 从数据库执行 SQL 查询并重组为 `Panel` 返回。

In [ ]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

FT = FDB.getTable("日行情表", args={"LookBack": 0})
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

# 因子表

In [ ]:
# 获取因子表对象 — 以日行情表（WideTable）为例
FT = FDB.getTable("日行情表", args={"LookBack": 0})
print(qs_help(FT))

In [ ]:
# 获取因子列表
print(FT.FactorNames)

# WideTable 详解

WideTable 是最常用的因子表类型，每行数据唯一对应一个（ID, 时点）组合，每个字段（除 ID 和时点字段外）是一个因子。

![Wide_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_Demo.png)

WideTable 的核心特性：

## 缺失填充（LookBack）

数据库中的数据常常是稀疏的（如非交易日无行情数据），`LookBack` 参数控制如何回填空缺。

In [ ]:
# LookBack=0：不填充，非交易日为 NaN
FT = FDB.getTable("日行情表", args={"LookBack": 0})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("LookBack=0 :")
print(Data)

In [ ]:
# LookBack>0：向前回溯填充缺失值，inf 表示无限回溯
FT = FDB.getTable("日行情表", args={"LookBack": 2})
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("LookBack=2 :")
print(Data)

## 公告时点（PublDTField）

对于财务数据，财报有报告期和公告日期之分。在公告日之前，即使报告期已过，也不应看到该数据（避免未来信息）。`PublDTField` 参数指定公告日期字段，确保数据只在公告后才能被读取到。

![Wide_Table_AnnDT_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_AnnDT_Demo.png)

In [ ]:
DTs = [dt.datetime(2023, 6, 30), dt.datetime(2023, 9, 30), dt.datetime(2023, 10, 1)]
IDs = ["000001.SZ"]

# 不考虑公告时点：2023-10-01 获取到了 2023-09-30 的数据（未来信息泄漏）
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable", "LookBack": np.inf, "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("不限制公告时点 :")
print(Data)

print("-" * 10)
# 考虑公告时点：2023-10-01 只能看到 2023-06-30 的数据（更早公告的）
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable", "LookBack": np.inf, "PublDTField": "信息发布日期",
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("限制公告时点 :")
print(Data)

## 多重映射（MultiMapping）

当一个（ID, 时点）对应多个数据行时（如同一报告期有多份修正财报），启用 `MultiMapping=True` 可将多值聚合为 list。可通过 `Operator` 参数指定聚合函数（如取均值）。

![Wide_Table_MultiMapping_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_MultiMapping_Demo.png)

In [ ]:
DTs = [dt.datetime(2023, 9, 30), dt.datetime(2023, 12, 31)]
IDs = ["000001.SZ"]

# MultiMapping=True：每个单元格是一个 list
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable", "LookBack": 0, "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"}, "MultiMapping": True
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("MultiMapping=True :")
print(Data)

print("-" * 10)
# 通过 Operator 取均值将多值合并为单一值
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable", "LookBack": 0, "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"}, "MultiMapping": True,
    "Operator": lambda x: x.mean(), "OperatorDataType": "double"
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("MultiMapping + Operator(mean) :")
print(Data)

# 其他常用因子表类型

## FeatureTable

继承自 WideTable，适用于**没有时点维度**的数据（如证券基本属性），或只取最新值的场景。默认 `LookBack=inf`，所有请求的时点都返回相同的数据。

![Feature_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Feature_Table_Demo.png)

In [ ]:
# FeatureTable：所有时点数据一样
FT = FDB.getTable("A股证券主表")
DTs = [dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)]
Data = FT.readData(factor_names=["证券简称", "上市日期"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

## TimeSeriesTable

无 ID 字段，只有时点字段的时序数据表。每个时点只有一行数据，所有 ID 共享同一套值。可以理解为 ID 字段为单一值的 WideTable。默认 `LookBack=inf`。

典型场景：宏观经济指标（如无风险利率、GDP 增速），对所有证券都相同。

![TimeSeries_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/TimeSeries_Table_Demo.png)

## NarrowTable

高瘦表（或叫"长表"），每行是一个四元组：**(ID, 时点, 因子名称, 因子值)**。通过 `FactorNameField` 指定因子名称字段，`FactorValueField` 指定因子值字段，系统将其 pivot 为宽表形式的因子数据。默认 `MultiMapping=True`。

![Narrow_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Narrow_Table_Demo.png)

## MappingTable

在**起始时点到截止时点之间**填充相同的值。典型应用于行业分类数据：某只股票在某段时间内属于某个行业，该时间段内所有时点都填充相同的行业代码。

核心参数：
| 参数 | 说明 |
|------|------|
| `DTField` | 起始时点字段 |
| `EndDTField` | 截止时点字段 |
| `EndDTIncluded` | 截止时点处是否填充数据（默认 `True`） |

![Mapping_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Mapping_Table_Demo.png)

In [ ]:
# MappingTable：在 DTField 和 EndDTField 之间的时点填充相同值
FT = FDB.getTable("公司行业划分表", args={"AdditionalCondition": {"行业划分标准": "37"}})
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["一级行业代码", "一级行业名称"], ids=["600136.SH"], dts=DTs).iloc[:, :, 0]
print(Data)

## ConstituentTable

成份股数据，因子值是一个 0/1 哑变量，1 表示该证券属于对应指数成份。`GroupField` 参数指定类别字段，其每个取值展开为一个因子。

核心参数：
| 参数 | 说明 |
|------|------|
| `DTField` | 入选日期字段 |
| `EndDTField` | 剔除日期字段 |
| `GroupField` | 类别字段（如"指数内部编码"），每个取值成为一个因子 |
| `EndDTIncluded` | 截止时点是否仍属于成份（默认 `False`） |
| `GroupMapping` | 类别值的名称映射（SQL 或 dict） |

![Constituent_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Constituent_Table_Demo.png)

In [ ]:
# ConstituentTable：成份区间内值为 1，其余为 0
FT = FDB.getTable("指数成份")
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["000016.SH", "000300.SH"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

## FinancialTable

最复杂的因子表类型，专门处理财务报表数据。财务数据的关键特征是每条记录有**报告期**和**公告日期**，需要根据当前时点判断应该使用哪个报告期的数据。

核心参数：

| 参数 | 取值 | 说明 |
|------|------|------|
| `CalcType` | `"最新"` / `"单季度"` / `"TTM"` | 财务数据转换成因子值的方式 |
| `ReportDate` | `"所有"` / `"定期报告"` / `"年报"` / `"中报"` / `"一季报"` / `"三季报"` | 使用的报告期类型 |
| `PublDTField` | `str \| None` | 公告日期字段，用于控制数据在公告后才可用 |
| `YearLookBack` | `int` | 回溯年数：取 n 年前同报告期的数据 |
| `PeriodLookBack` | `int` | 回溯期数：取 n 期前同报告期的数据 |

**CalcType 详解：**

- **最新**：当前时点能看到的最新报告期数据。例如 2010-08-20 时某公司已披露 2010 年中报，则用中报数据
- **单季度**：当前时点能看到的单季度数据。例如 2010-08-20 已披露中报 → 净利润(中报) - 净利润(一季报) = Q2 净利润
- **TTM**：当前时点能看到的滚动四季度数据。例如 2010-08-20 已披露中报 → 净利润(中报) + 净利润(去年年报) - 净利润(去年中报)

![Financial_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Financial_Table_Demo.png)

In [ ]:
# FinancialTable：CalcType="最新"，ReportDate="年报"
FT = FDB.getTable("资产负债表_新会计准则", args={
    "ReportDate": "年报",
    "CalcType": "最新"
})
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["资产总计"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

## MacroTable

专门处理宏观数据。宏观数据的关键特征是每条记录有**截止期**和**公告日期**，需要根据当前时点判断应该使用哪个截止期的数据。

另外，早期的公告日期可能缺失或者不可靠，需要根据可靠的公告日期数据对其修正。

核心参数：

| 参数 | 取值 | 说明 |
|------|------|------|
| `PublDTCleanFunc` | `Callable` | 公告时点修正算子 |
| `PublDTField` | `str \| None` | 公告日期字段，用于控制数据在公告后才可用 |
| `PeriodLookBack` | `int` | 回溯期数：取 n 期前同报告期的数据 |

![Wide_Table_AnnDT_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_AnnDT_Demo.png)

In [10]:
FT = FDB.getTable("宏观基础指标数据", args={"PeriodLookBack": 2, "LookBack": np.inf})
DTs = [dt.datetime(2026, 1, 16), dt.datetime(2026, 2, 16), dt.datetime(2026, 7, 14)]
Data = FT.readData(factor_names=["指标数据"], ids=["110251594"], dts=DTs).iloc[0]
print(Data)

                     110251594
2026-01-16 00:00:00      -13.9
2026-02-16 00:00:00      -14.7
2026-07-14 00:00:00      -11.2


# 因子

通过 `FT.getFactor()` 获取的因子对象是通用的 `Factor` 实例，与其他因子库中的因子 API 完全一致。详细的因子操作（运算符重载、衍生因子、readData 等）请参见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。

In [ ]:
# 获取因子并读取数据
F = FT.getFactor("资产总计")
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs)
print(Data)